- Load the raw data from the storage to the bronze layer of the data lakehouse

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("bronze_loading").getOrCreate()

In [0]:
spark

- Define the schema for the data to be loaded

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

teleco_customer_churn_schema = StructType([
    StructField("customerID", StringType()),
    StructField("count",IntegerType()),
    StructField("country", IntegerType()),
    StructField("state", StringType()),
    StructField("City", StringType()),
    StructField("zip_code", StringType()),
    StructField("latlong", DoubleType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("gender", StringType()),
    StructField("senior_citizen", StringType()),
    StructField("patner", StringType()),
    StructField("dependants", StringType()),
    StructField("tenure_months", IntegerType()),
    StructField("phone_service", StringType()),
    StructField("multiple_lines", StringType()),
    StructField("internet_service", StringType()),
    StructField("online_security", StringType()),
    StructField("online_backup", StringType()),
    StructField("device_protection_plan", StringType()),
    StructField("tech_support", StringType()),
    StructField("streaming_tv", StringType()),
    StructField("streaming_movies", StringType()),
    StructField("contract", StringType()),
    StructField("paperless_billing", StringType()),
    StructField("payment_method", StringType()),
    StructField("monthly_charges", DoubleType()),
    StructField("total_charges", DoubleType()),
    StructField("churn_label", StringType()),
    StructField("churn_value", IntegerType()),
    StructField("churn_score", IntegerType()),
    StructField("cltv", IntegerType()),
    StructField("churn_reason", StringType())

])

- Load the data as a pandas dataframe before converting it to a spark dataframe.

In [0]:
# Install openpyxl
%pip install openpyxl

In [0]:
# Confirm that openpyxl exists
!pip show openpyxl

In [0]:
import pandas as pd

excel_file_path = "/Volumes/teleco_churn_datalakehouse/bronze/raw_sources/Telco_customer_churn.xlsx"
pdf = pd.read_excel(excel_file_path, sheet_name='Telco_Churn', engine='openpyxl')


pdf['Total Charges'] = pd.to_numeric(pdf['Total Charges'], errors='coerce')


df = spark.createDataFrame(pdf)


df.display()

- Replacing blank spaces, forward slashes and dashes with an underscore in column names

In [0]:
for col in df.columns:
    new_col = col.replace(' ', '_').replace('/', '_').replace('-', '_')
    df = df.withColumnRenamed(col, new_col)

# Specify the database/schema when saving the table
df.write.mode("overwrite").saveAsTable("teleco_churn_datalakehouse.bronze.teleco_customer_churn")

- Check for the data loaded into a table in the bronze layer

In [0]:
%sql
SELECT * FROM
teleco_churn_datalakehouse.bronze.teleco_customer_churn; 